# **Feed Forward Network (FFN)**

## What is the Feed-Forward Network (FFN)?

Self-attention lets words talk to each other — it's about relationships. But after that conversation, each word needs time to think individually — to process and digest what it just learned. That's what FFN does.

Think of it like a classroom:
```
Attention = group discussion (students share information)
FFN       = individual thinking (each student processes what they heard)
```

The FFN is applied to each token independently — same formula, same weights, but each word goes through it separately. It doesn't look at other words. That's the key difference from attention.

The formula is simple:
```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

That's just two linear layers with a ReLU activation in between.

```
Step 1: x @ W1 + b1        → expand to bigger dimension
Step 2: ReLU()              → kill negative values (non-linearity!)
Step 3: result @ W2 + b2   → compress back to original dimension
```

Why expand then compress?
The standard pattern is to expand to `4x the size`, then compress back:
`d_model = 8  →  expand to 32 (8 × 4)  →  compress back to 8`

```
Like taking a deep breath:
  inhale (expand): give the model MORE space to think, more neurons to work with
  exhale (compress): squeeze the important stuff back into original size
```

This expansion gives the network a bigger "workspace" to compute complex transformations. In real models:

```
GPT-2: d_model=768   → expand to 3072 (768 × 4) → back to 768
GPT-3: d_model=12288 → expand to 49152           → back to 12288
```

Why ReLU?
To solve the `linearity problem`: Two linear layers stacked without activation is just one linear layer — useless stacking. ReLU breaks linearity so the network can learn complex patterns. Without it:
```
(x @ W1) @ W2 = x @ (W1 @ W2) = x @ W_combined
                                  ↑
                        Just ONE linear layer! No benefit from two.
                        
With ReLU between them: can't simplify, genuinely two layers, more power.
```


## What ReLU actually does?

In [ ]:
# Let's see ReLU in action
import torch

sample = torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.5])
print("Before ReLU:", sample)
print("After ReLU: ", torch.relu(sample))
# [-2.0, -0.5, 0.0, 0.3, 1.5]  →  [0.0, 0.0, 0.0, 0.3, 1.5]
# Negatives become zero, positives stay unchanged

Before ReLU: tensor([-2.0000, -0.5000,  0.0000,  0.3000,  1.5000])
After ReLU:  tensor([0.0000, 0.0000, 0.0000, 0.3000, 1.5000])


# FFN - The Manual Way

In [ ]:
import torch
import math

torch.manual_seed(42)

# Fake input: 7 tokens, d_model=8 (pretend this came from attention)
x = torch.randn(7, 8)

d_model = 8
d_ff = 32   # expand to 4x (8 × 4 = 32)

# Create weight matrices manually
W1 = torch.randn(d_model, d_ff)    # (8, 32) — expand
b1 = torch.zeros(d_ff)             # (32,)

W2 = torch.randn(d_ff, d_model)    # (32, 8) — compress back
b2 = torch.zeros(d_model)          # (8,)

# Forward pass — step by step
print("Input shape:", x.shape)                     # (7, 8)

# Step 1: First linear layer (expand)
hidden = x @ W1 + b1
print("After expand:", hidden.shape)                # (7, 32)

# Step 2: ReLU activation (kill negatives)
hidden = torch.relu(hidden)
print("After ReLU:", hidden.shape)                  # (7, 32)
# Some values are now 0 (the negative ones died)

# Step 3: Second linear layer (compress back)
output = hidden @ W2 + b2
print("After compress:", output.shape)              # (7, 8)

print("\nInput shape == Output shape?", x.shape == output.shape)  # True!

Input shape: torch.Size([7, 8])
After expand: torch.Size([7, 32])
After ReLU: torch.Size([7, 32])
After compress: torch.Size([7, 8])

Input shape == Output shape? True


## FFN - PyTorch Way

Same math as the manual version, but PyTorch manages everything. Compare the two:

```
Manual:                          PyTorch:
W1 = torch.randn(8, 32)      →    self.layer1 = nn.Linear(8, 32)
b1 = torch.zeros(32)         →    (bias included automatically)
hidden = x @ W1 + b1         →    x = self.layer1(x)
hidden = torch.relu(hidden)  →    x = self.relu(x)
W2 = torch.randn(32, 8)      →    self.layer2 = nn.Linear(32, 8)
output = hidden @ W2 + b2    →    x = self.layer2(x)
```


In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two linear layers — expand then compress
        # nn.Linear handles weights AND biases automatically
        self.layer1 = nn.Linear(d_model, d_ff)       # (8 → 32) expand
        self.relu = nn.ReLU()                        # activation
        self.layer2 = nn.Linear(d_ff, d_model)       # (32 → 8) compress

    def forward(self, x):
        # Each step on its own line for clarity
        x = self.layer1(x)    # expand:   (7, 8)  → (7, 32)
        x = self.relu(x)      # activate: (7, 32) → (7, 32) (negatives → 0)
        x = self.layer2(x)    # compress: (7, 32) → (7, 8)
        return x

# Create and run
ffn = FeedForward(d_model=8, d_ff=32)
x = torch.randn(7, 8)
output = ffn(x)

print("Input shape:", x.shape)       # (7, 8)
print("Output shape:", output.shape)  # (7, 8)

# Let's see how many parameters this has
total_params = sum(p.numel() for p in ffn.parameters())
print(f"\nTotal parameters: {total_params}")
# W1: 8×32=256, b1: 32, W2: 32×8=256, b2: 8 → total = 552

Input shape: torch.Size([7, 8])
Output shape: torch.Size([7, 8])

Total parameters: 552


## How FFN Works?

### Tokens and Neurons:

The 7 tokens are NOT 7 neurons. Tokens and neurons are different things.
```
Tokens = your DATA (the 7 words flowing through the network)
Neurons = the MACHINERY (the processing units inside each layer)
```
Think of it like a factory:
```
Tokens  = 7 items on a conveyor belt (they move through)
Neurons = the machines in the factory (they stay fixed, process each item)
```
The number of neurons in a layer is determined by the output dimension, not by how many tokens you have. Whether you feed 7 tokens or 700 tokens, the FFN has the same number of neurons.

---

### Let's build up from a single neuron
One neuron does this:
```
output = activation(input₁×w₁ + input₂×w₂ + ... + input₈×w₈ + bias)
```
It takes ALL inputs (8 numbers), multiplies each by its own weight, sums them up, adds bias, applies activation. One neuron produces one number.

Let's say our first token "i" has embedding `[0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6]` — that's 8 numbers.

One neuron processing this token:
```
neuron_1 output = ReLU(0.2×w₁ + 0.5×w₂ + 0.1×w₃ + 0.8×w₄
                     + 0.3×w₅ + 0.7×w₆ + 0.4×w₇ + 0.6×w₈ + bias)
               = ReLU(some single number)
               = one number
```               
But we need 32 output numbers (expanding to d_ff=32). So we need 32 neurons, each with its own set of 8 weights:
```
neuron_1:  ReLU(0.2×w₁₁ + 0.5×w₁₂ + ... + 0.6×w₁₈ + b₁) = out₁
neuron_2:  ReLU(0.2×w₂₁ + 0.5×w₂₂ + ... + 0.6×w₂₈ + b₂) = out₂
neuron_3:  ReLU(0.2×w₃₁ + 0.5×w₃₂ + ... + 0.6×w₃₈ + b₃) = out₃
...
neuron_32: ReLU(0.2×w₃₂,₁ + 0.5×w₃₂,₂ + ... + 0.6×w₃₂,₈ + b₃₂) = out₃₂
```
32 neurons, each producing one number → output is 32 numbers. That's our expansion from 8 to 32.

---

### 32 Neurons with Inputs:

Let's write all 32 neurons side by side. The weights of all neurons form a matrix:
```
W1 = [ neuron_1 weights:  w₁₁, w₁₂, w₁₃, w₁₄, w₁₅, w₁₆, w₁₇, w₁₈ ]
     [ neuron_2 weights:  w₂₁, w₂₂, w₂₃, w₂₄, w₂₅, w₂₆, w₂₇, w₂₈ ]
     [ neuron_3 weights:  w₃₁, w₃₂, w₃₃, w₃₄, w₃₅, w₃₆, w₃₇, w₃₈ ]
     ...
     [ neuron_32 weights: w₃₂,₁ ...                         w₃₂,₈]

Shape: (32, 8) — 32 neurons, each with 8 weights
```

And the bias of all neurons forms a vector:
```
b1 = [b₁, b₂, b₃, ... b₃₂]

Shape: (32,)
```
Now when we compute x @ W1.T + b1 for one token:
```
input:  [0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6]    shape: (8,)
W1.T:   (8, 32)     ← transposed so dimensions align
result: (32,)        ← one output per neuron

input @ W1.T = [
    0.2×w₁₁ + 0.5×w₁₂ + ... + 0.6×w₁₈,    ← neuron 1's sum
    0.2×w₂₁ + 0.5×w₂₂ + ... + 0.6×w₂₈,    ← neuron 2's sum
    ...
    0.2×w₃₂,₁ + 0.5×w₃₂,₂ + ... + 0.6×w₃₂,₈  ← neuron 32's sum
]

+ b1 = add each neuron's bias

ReLU() = apply activation to each neuron's output
```
That's it. Matrix multiplication is just running all neurons simultaneously. Each row of W1 is one neuron's weights. The matrix multiply computes all 32 neurons in one shot.
```
Writing 32 neurons individually:
  neuron_k output = ReLU(input₁×wₖ₁ + input₂×wₖ₂ + ... + input₈×wₖ₈ + bₖ)

Writing them all at once (matrix form):
  all outputs = ReLU(x @ W1.T + b1)

SAME MATH. Matrix form is just the shorthand.
```

---

### Now what about 7 tokens?
We have 7 tokens, each with 8 dimensions. We stack them into a matrix:
```
x = [ token_0: [0.2, 0.5, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6] ]
    [ token_1: [0.9, 0.1, 0.4, 0.3, 0.8, 0.2, 0.5, 0.7] ]
    [ token_2: [0.4, 0.6, 0.8, 0.1, 0.5, 0.9, 0.2, 0.3] ]
    ...
    [ token_6: [0.7, 0.3, 0.6, 0.5, 0.1, 0.4, 0.8, 0.2] ]

Shape: (7, 8)
When we compute x @ W1.T + b1:
(7, 8) @ (8, 32) + (32,) = (7, 32)

Row 0: token_0 goes through all 32 neurons → 32 outputs
Row 1: token_1 goes through all 32 neurons → 32 outputs
Row 2: token_2 goes through all 32 neurons → 32 outputs
...
Row 6: token_6 goes through all 32 neurons → 32 outputs
```
Every token goes through the SAME 32 neurons independently. The matrix multiplication handles all 7 tokens at once, but each token is processed by the same set of neurons with the same weights. Token 0 doesn't affect token 1's FFN computation.

---

### The full FFN with neuron mapping

Now let's trace the complete `FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2` and map it to neurons:
```
LAYER 1: 32 neurons (expanding)
┌───────────────────────────────────────────────────┐
│ Each of the 32 neurons takes 8 inputs             │
│ Each neuron has 8 weights + 1 bias                │
│ All weights together = W1 matrix (32, 8)          │
│ All biases together = b1 vector (32,)             │
│                                                   │
│ For each token independently:                     │
│   step 1: x @ W1.T + b1    (compute all 32 sums)  │
│   step 2: ReLU()            (kill negatives)      │
│   output: 32 numbers per token                    │
│                                                   │
│ input shape:  (7, 8)                              │
│ output shape: (7, 32)                             │
└───────────────────────────────────────────────────┘
                    ↓

LAYER 2: 8 neurons (compressing)
┌──────────────────────────────────────────────────┐
│ Each of the 8 neurons takes 32 inputs            │
│ Each neuron has 32 weights + 1 bias              │
│ All weights together = W2 matrix (8, 32)         │
│ All biases together = b2 vector (8,)             │
│                                                  │
│ For each token independently:                    │
│   result @ W2.T + b2   (compute all 8 sums)      │
│   No activation here (output goes to next layer) │
│   output: 8 numbers per token                    │
│                                                  │
│ input shape:  (7, 32)                            │
│ output shape: (7, 8)                             │
└──────────────────────────────────────────────────┘
```
Summary of the neuron structure:
```
Layer 1: 32 neurons, each takes 8 inputs  → total weights: 32 × 8 = 256
Layer 2: 8 neurons,  each takes 32 inputs → total weights: 8 × 32 = 256
Biases:  32 + 8 = 40
Total parameters: 256 + 256 + 40 = 552
```

---

### Connecting the two formulas:
```
Single neuron formula:
  output = activation(input × weight + bias)

FFN matrix formula:
  FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2

The connection:
  x @ W1 + b1     = running ALL layer-1 neurons at once
  ReLU(...)        = applying activation to all neurons at once  
  ... @ W2 + b2   = running ALL layer-2 neurons at once

Matrix multiplication IS neurons running in parallel.
The formulas are identical — one is zoomed into a single neuron,
the other is zoomed out to see all neurons at once.
```